# ReAct Research Assistant

The ReAct (Reasoning + Acting) loop is the workhorse pattern behind most research agents. The LLM alternates between forming a thought, choosing a tool, observing the result, and repeating — until it can write a final answer grounded in real sources.

**What you'll build:** a research assistant that searches DuckDuckGo for current news, Wikipedia for background knowledge, and arXiv for academic papers — with a step budget and conversation memory so it stays on topic.

**Time:** ~25 min | **Difficulty:** Intermediate

**What you'll learn:**
- How the Thought → Action → Observation loop works internally
- When to use `ReActAgent` vs `FunctionCallingAgent`
- How to combine `DuckDuckGoSearchTool`, `WikipediaTool`, and `ArxivSearchTool`
- How to cap token spend with `max_iterations`
- How to inspect the reasoning trace via `agent.memory`

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package (installed in the next cell)

Set your API key in the environment setup cell below before running any agent cells.

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key here before running the cells below
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Install and import

Import the `ReActAgent`, the three search tools, `AgentMemory` for tracking steps, and the OpenAI LLM wrapper.

In [ ]:
import asyncio
from synapsekit.agents import (
    ReActAgent,
    ArxivSearchTool,
    DuckDuckGoSearchTool,
    WikipediaTool,
    AgentMemory,
)
from synapsekit.llms.openai import OpenAILLM

## Step 2: Create the tools

Each tool has a single `run(input: str)` method. The agent selects which tool to call based on the tool's `name` and `description` — those strings appear verbatim in the system prompt, so make them precise.

- `DuckDuckGoSearchTool` — current events, news
- `WikipediaTool` — encyclopedic background
- `ArxivSearchTool` — academic papers

In [ ]:
tools = [
    DuckDuckGoSearchTool(),   # current events, news
    WikipediaTool(),           # encyclopedic background
    ArxivSearchTool(),         # academic papers
]

## Step 3: Configure the agent

`max_iterations` acts as a step budget. Each Thought/Action/Observation cycle counts as one iteration. Setting it to 8 means the agent can call up to 8 tools before being forced to answer.

In [ ]:
memory = AgentMemory(max_steps=8)

agent = ReActAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=tools,
    max_iterations=8,
    memory=memory,
)

## Step 4: Run a research query

`agent.run()` blocks until the agent produces a final answer. Use `await` inside an async function.

In [ ]:
async def research(question: str) -> str:
    return await agent.run(question)

# Test it
answer = await research("What is quantum entanglement?")
print(answer)

## Step 5: Inspect the reasoning trace

After `agent.run()` completes, `agent.memory.steps` contains every Thought, Action, and Observation the agent produced. This is useful for debugging why the agent chose a particular path.

In [ ]:
def print_trace(agent: ReActAgent) -> None:
    for i, step in enumerate(agent.memory.steps, start=1):
        print(f"Step {i}")
        print(f"  Thought:     {step.thought}")
        print(f"  Action:      {step.action}")
        print(f"  Input:       {step.action_input[:80]}...")
        print(f"  Observation: {step.observation[:120]}...")
        print()

print_trace(agent)

## Step 6: Reuse memory across turns

Clearing memory between unrelated questions prevents the scratchpad from bleeding context across sessions. For a multi-turn research session on the same topic, skip the clear so the agent retains prior observations.

In [ ]:
async def research_session(questions: list[str]) -> None:
    for question in questions:
        # Clear between unrelated questions; remove this line for follow-up questions
        agent.memory.clear()
        answer = await agent.run(question)
        print(f"Q: {question}")
        print(f"A: {answer}\n")

await research_session([
    "What is the Higgs boson?",
    "Who invented the World Wide Web?",
])

## Step 7: Stream step events for interactive display

Instead of waiting for the full answer, stream each step event so users see the agent "thinking" in real time.

In [ ]:
from synapsekit.agents import ThoughtEvent, ActionEvent, ObservationEvent, FinalAnswerEvent

async def stream_research(question: str) -> None:
    async for event in agent.stream_steps(question):
        if isinstance(event, ThoughtEvent):
            print(f"Thinking: {event.thought}")
        elif isinstance(event, ActionEvent):
            print(f"Calling:  {event.tool}({event.tool_input!r})")
        elif isinstance(event, ObservationEvent):
            print(f"Result:   {event.observation[:100]}...")
        elif isinstance(event, FinalAnswerEvent):
            print(f"\nFinal answer:\n{event.answer}")

agent.memory.clear()
await stream_research("What are the latest developments in fusion energy?")

## Complete Working Example

A self-contained script that builds a fresh agent, streams step events for a research question, and prints the full reasoning trace at the end. Wrapped in `asyncio.run(main())` so it works in Colab and as a standalone script.

In [ ]:
import asyncio
from synapsekit.agents import (
    ReActAgent,
    ArxivSearchTool,
    DuckDuckGoSearchTool,
    WikipediaTool,
    AgentMemory,
    ThoughtEvent,
    ActionEvent,
    ObservationEvent,
    FinalAnswerEvent,
)
from synapsekit.llms.openai import OpenAILLM


def build_agent() -> ReActAgent:
    return ReActAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[
            DuckDuckGoSearchTool(),
            WikipediaTool(),
            ArxivSearchTool(),
        ],
        max_iterations=8,
        memory=AgentMemory(max_steps=8),
    )


async def main() -> None:
    agent = build_agent()
    question = (
        "What are the most recent breakthroughs in large language model alignment? "
        "Include at least one academic paper and one news item."
    )

    print(f"Question: {question}\n")
    print("=" * 60)

    async for event in agent.stream_steps(question):
        if isinstance(event, ThoughtEvent):
            print(f"[Thought]      {event.thought}")
        elif isinstance(event, ActionEvent):
            print(f"[Action]       {event.tool} <- {str(event.tool_input)[:80]}")
        elif isinstance(event, ObservationEvent):
            print(f"[Observation]  {event.observation[:120]}")
        elif isinstance(event, FinalAnswerEvent):
            print("\n" + "=" * 60)
            print("FINAL ANSWER")
            print("=" * 60)
            print(event.answer)

    print("\n--- Reasoning trace ---")
    for i, step in enumerate(agent.memory.steps, start=1):
        print(f"Step {i}: {step.action}({step.action_input[:60]})")


asyncio.run(main())

## Next Steps

- [Streaming Agent Responses](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/streaming-agent) — display thought events in a terminal UI or WebSocket
- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — add five or more tools and handle parallel calls
- [Agent with Safety Guardrails](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/agent-with-guardrails) — validate inputs and outputs before and after each run